# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2f47b69d-fb10-45a4-99cc-d7907c3b4e97;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses *(À compléter)* :**
- **Test #1 :** $H_0 =$ `...` / $H_1 =$ `...` *(Justification du choix du test : `...`)*
- **Test #2 :** $H_0 =$ `...` / $H_1 =$ `...` *(Justification du choix du test : `...`)*

In [5]:
from pyspark.sql import functions as F

silver_path = "/lakehouse/silver"

fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
)
path = spark._jvm.org.apache.hadoop.fs.Path(silver_path)

for item in fs.listStatus(path):
    print(item.getPath().toString())

hdfs://namenode:9000/lakehouse/silver/olist_order_items_enriched
hdfs://namenode:9000/lakehouse/silver/olist_order_items_merge_test


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [ ]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/..."  # TODO : Chemin de votre table Gold de segmentation

# TODO 1 : Agréger les features par entité métier côté PySpark puis convertir la table d'entités en Pandas
entities_pd = None
assert entities_pd is not None, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."

# TODO 2 : Prétraiter (log1p + StandardScaler), entraîner KMeans et nommer métier chaque cluster dans `entities_pd["segment"]`
...

assert "segment" in entities_pd.columns, "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."

# TODO 3 : Sauvegarder en Delta Lake sur HDFS dans `GOLD_SEGMENTS_PATH`
(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} entités) !")

## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [20]:
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
)
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory()
    and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]

assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")

for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(
        f"/lakehouse/gold/{t}"
    ).count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/gold_monthly_evolution    :        24 lignes


 - /lakehouse/gold/gold_top_month_by_status  :         7 lignes


[Stage 141:====================================================>  (48 + 2) / 50]

 - /lakehouse/gold/olist_customer_segments   :    93,350 lignes


In [1]:
# 1. Importer les bibliothèques
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 2. Initialiser Spark
spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

# 3. Charger notre table Silver Olist
SILVER_MAIN_PATH = "/lakehouse/silver/olist_order_items_enriched"

df_silver = spark.read.format("delta").load(SILVER_MAIN_PATH)

# 4. Vérifier les colonnes
df_silver.printSchema()

# 5. Afficher quelques lignes
df_silver.show(5, truncate=False)

print("Nombre de lignes :", df_silver.count())

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7795ccd4-b332-4c47-8289-c5d1e330d41a;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- item_total_amount: double (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)



26/11/17 21:23:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--------------------------+-------------------+-------------------------------------------------------------------------------------------+--------------------------------+------------+------------------------+-----------------------------+------------------+-------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at              |_source_dataset    |_source_file                                                                               |customer_id                     |order_status|order_purchase_timestamp|order_delivered_customer_date|item_total_amount |delivery_delay_days|
+--------------------------------+-------------+--------------------------------+--------------------------------+------------------

[Stage 11:======================================================> (49 + 1) / 50]

Nombre de lignes : 112650


In [2]:
print("Spark version :", spark.version)
print("Application ID :", spark.sparkContext.applicationId)
print("Master :", spark.sparkContext.master)

spark.range(5).show()

Spark version : 4.2.0
Application ID : local-1795104854360
Master : local[*]


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [3]:
spark.range(5).show()


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [6]:
SILVER_MAIN_PATH = "hdfs://namenode:9000/lakehouse/silver/olist_order_items_enriched"

df_silver = spark.read.format("delta").load(SILVER_MAIN_PATH)

print("Nombre de lignes :", df_silver.count())
print("Nombre de colonnes :", len(df_silver.columns))

df_silver.printSchema()
df_silver.show(5, truncate=False)

26/11/19 16:32:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Nombre de lignes : 112650
Nombre de colonnes : 16
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- item_total_amount: double (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)



[Stage 16:======================================>                   (2 + 1) / 3]

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--------------------------+-------------------+-------------------------------------------------------------------------------------------+--------------------------------+------------+------------------------+-----------------------------+------------------+-------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at              |_source_dataset    |_source_file                                                                               |customer_id                     |order_status|order_purchase_timestamp|order_delivered_customer_date|item_total_amount |delivery_delay_days|
+--------------------------------+-------------+--------------------------------+--------------------------------+------------------

In [7]:
# Répartition des commandes par statut
df_silver.groupBy("order_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

# Statistiques descriptives
df_silver.select(
    "price",
    "freight_value",
    "item_total_amount",
    "delivery_delay_days"
).describe().show()

# Valeurs manquantes
df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "price",
        "freight_value",
        "item_total_amount",
        "delivery_delay_days"
    ]
]).show()

+------------+------+
|order_status| count|
+------------+------+
|   delivered|110197|
|     shipped|  1185|
|    canceled|   542|
|    invoiced|   359|
|  processing|   357|
| unavailable|     7|
|    approved|     3|
+------------+------+



+-------+------------------+------------------+------------------+-------------------+
|summary|             price|     freight_value| item_total_amount|delivery_delay_days|
+-------+------------------+------------------+------------------+-------------------+
|  count|            112650|            112650|            112650|             110196|
|   mean|120.65373901477298| 19.99031992898563|140.64405894362073| 12.410958655486587|
| stddev| 183.6339280502588|15.806405412297153| 190.7243940317416|    9.4551469219301|
|    min|              0.85|               0.0|              6.08|                  0|
|    max|            6735.0|            409.68|           6929.31|                210|
+-------+------------------+------------------+------------------+-------------------+



+-----+-------------+-----------------+-------------------+
|price|freight_value|item_total_amount|delivery_delay_days|
+-----+-------------+-----------------+-------------------+
|    0|            0|                0|               2454|
+-----+-------------+-----------------+-------------------+



In [8]:
# Garder uniquement les commandes livrées et annulées
df_stats = df_silver.filter(
    F.col("order_status").isin("delivered", "canceled")
)

# Sélectionner uniquement les colonnes nécessaires
df_stats = df_stats.select(
    "order_status",
    "price",
    "freight_value"
).dropna()

# Préparer un échantillon pour les tests statistiques
stats_pd = df_stats.toPandas()

# Séparer les deux groupes
prices_delivered = stats_pd.loc[
    stats_pd["order_status"] == "delivered", "price"
]

prices_canceled = stats_pd.loc[
    stats_pd["order_status"] == "canceled", "price"
]

print("Articles livrés :", len(prices_delivered))
print("Articles annulés :", len(prices_canceled))
print("Échantillon total :", len(stats_pd))

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

Articles livrés : 110197
Articles annulés : 542
Échantillon total : 110739


In [9]:
# TEST 1 : Mann-Whitney U
mann_result = stats.mannwhitneyu(
    prices_delivered,
    prices_canceled,
    alternative="two-sided"
)

print("=== TEST DE MANN-WHITNEY ===")
print("Statistique U :", mann_result.statistic)
print("P-value :", mann_result.pvalue)

if mann_result.pvalue < 0.05:
    print("Conclusion : on rejette H0.")
    print("Les distributions de prix sont significativement différentes.")
else:
    print("Conclusion : on ne rejette pas H0.")


# TEST 2 : Corrélation de Spearman
spearman_result = stats.spearmanr(
    stats_pd["price"],
    stats_pd["freight_value"]
)

print("\n=== CORRÉLATION DE SPEARMAN ===")
print("Coefficient rho :", spearman_result.statistic)
print("P-value :", spearman_result.pvalue)

if spearman_result.pvalue < 0.05:
    print("Conclusion : la corrélation est statistiquement significative.")
else:
    print("Conclusion : aucune corrélation significative détectée.")

=== TEST DE MANN-WHITNEY ===
Statistique U : 27847873.5
P-value : 0.006630237933591834
Conclusion : on rejette H0.
Les distributions de prix sont significativement différentes.

=== CORRÉLATION DE SPEARMAN ===
Coefficient rho : 0.4342574090691425
P-value : 0.0
Conclusion : la corrélation est statistiquement significative.


In [10]:
# Utiliser toutes les lignes de la table Silver
df_spearman = df_silver.select(
    "price",
    "freight_value"
).dropna()

spearman_pd = df_spearman.toPandas()

spearman_result = stats.spearmanr(
    spearman_pd["price"],
    spearman_pd["freight_value"]
)

print("Nombre de lignes :", len(spearman_pd))
print("Rho :", spearman_result.statistic)
print("P-value :", spearman_result.pvalue)

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

Nombre de lignes : 112650
Rho : 0.434190324041091
P-value : 0.0


In [11]:
# 1. Regrouper les articles par commande
orders_df = (
    df_silver
    .filter(F.col("order_status") == "delivered")
    .groupBy(
        "customer_id",
        "order_id"
    )
    .agg(
        F.sum("item_total_amount").alias("order_amount"),
        F.max("delivery_delay_days").alias("delivery_delay")
    )
)

# 2. Construire les indicateurs par client
customers_df = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.sum("order_amount").alias("total_spent"),
        F.countDistinct("order_id").alias("number_orders"),
        F.avg("order_amount").alias("avg_order_value"),
        F.avg("delivery_delay").alias("avg_delivery_delay")
    )
    .dropna()
)

print("Nombre de clients :", customers_df.count())

customers_df.show(10, truncate=False)

Nombre de clients : 96470


[Stage 57:>                                                         (0 + 1) / 1]

+--------------------------------+-----------+-------------+---------------+------------------+
|customer_id                     |total_spent|number_orders|avg_order_value|avg_delivery_delay|
+--------------------------------+-----------+-------------+---------------+------------------+
|2daf50201a5e309f6be0fcb20ff2ddd5|175.16     |1            |175.16         |17.0              |
|9a5d390630499694895f2d2357a9967e|243.56     |1            |243.56         |55.0              |
|c79c8d356d92bb1b74efb6c4a7766527|115.55     |1            |115.55         |5.0               |
|36a1aa63bf2ebcd4911e026092700610|98.8       |1            |98.8           |3.0               |
|0721e1c4b91bc6ded6579edace8659d3|107.78     |1            |107.78         |12.0              |
|088935be5a63fd4166d69f633d2d1156|83.34      |1            |83.34          |20.0              |
|1187acd131fe97cfc7cde033a4f036dd|46.11      |1            |46.11          |45.0              |
|80447be02d8f3f3f309c7df938a08501|41.0  

In [12]:
# Afficher les tables disponibles dans Silver
for item in fs.listStatus(
    spark._jvm.org.apache.hadoop.fs.Path("/lakehouse/silver")
):
    print(item.getPath().toString())

hdfs://namenode:9000/lakehouse/silver/olist_order_items_enriched
hdfs://namenode:9000/lakehouse/silver/olist_order_items_merge_test


In [13]:
customers_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/lakehouse/bronze/raw/brazilian-ecommerce/olist_customers_dataset.csv")
)

customers_raw.printSchema()
customers_raw.select(
    "customer_id",
    "customer_unique_id"
).show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+
|customer_id                     |customer_unique_id              |
+--------------------------------+--------------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|
|4f2d8ab171c80ec8364f7c12e35b23ad|345ecd01c38d18a9036ed96c73b8d066|
+--------------------------------+--------------------------------+
only showing top 5 rows


In [14]:
# 1. Récupérer l'identifiant unique du client
orders_with_unique_id = orders_df.join(
    customers_raw.select(
        "customer_id",
        "customer_unique_id"
    ),
    on="customer_id",
    how="inner"
)

# 2. Agréger les données par client unique
customers_df = (
    orders_with_unique_id
    .groupBy("customer_unique_id")
    .agg(
        F.sum("order_amount").alias("total_spent"),
        F.countDistinct("order_id").alias("number_orders"),
        F.avg("order_amount").alias("avg_order_value"),
        F.avg("delivery_delay").alias("avg_delivery_delay")
    )
    .dropna()
)

print("Nombre de clients uniques :", customers_df.count())

customers_df.show(10, truncate=False)

Nombre de clients uniques : 93350


[Stage 87:=============================>                            (2 + 2) / 4]

+--------------------------------+------------------+-------------+------------------+------------------+
|customer_unique_id              |total_spent       |number_orders|avg_order_value   |avg_delivery_delay|
+--------------------------------+------------------+-------------+------------------+------------------+
|000c8bdb58a29e7115cfc257230fb21b|29.0              |1            |29.0              |15.0              |
|a695332469eaed4d7e8af674d1c0a7fb|69.5              |1            |69.5              |8.0               |
|aefbb459b35ac7289f8ed198bb4e5a18|57.06             |1            |57.06             |23.0              |
|71430416a809573ed4a01e04fa8e3b91|27.240000000000002|1            |27.240000000000002|11.0              |
|762efb8c186a318d9d711692c0030c85|146.57999999999998|1            |146.57999999999998|12.0              |
|c2676cdedacba0e3aed5312e06b9bdda|48.09             |1            |48.09             |13.0              |
|45ba2acc9e8cbf07b686d5994d7592ab|73.37       

In [15]:
# 1. Convertir les données agrégées en Pandas
entities_pd = customers_df.toPandas()

# 2. Définir les features
features = [
    "total_spent",
    "number_orders",
    "avg_order_value",
    "avg_delivery_delay"
]

# 3. Appliquer log1p aux variables positives
entities_pd[features] = np.log1p(entities_pd[features])

# 4. Standardiser les variables
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    entities_pd[features]
)

print("Nombre de clients :", len(entities_pd))
print("Nombre de features :", len(features))
print("Dimensions après standardisation :", X_scaled.shape)

print("\nAperçu des données transformées :")
print(entities_pd[features].head())

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

Nombre de clients : 93350
Nombre de features : 4
Dimensions après standardisation : (93350, 4)

Aperçu des données transformées :
   total_spent  number_orders  avg_order_value  avg_delivery_delay
0     3.401197       0.693147         3.401197            2.772589
1     4.255613       0.693147         4.255613            2.197225
2     4.061477       0.693147         4.061477            3.178054
3     3.340739       0.693147         3.340739            2.484907
4     4.994370       0.693147         4.994370            2.564949


In [16]:
# Repartir des données originales pour conserver
# les vraies valeurs lors de l'interprétation
entities_pd = customers_df.toPandas()

features = [
    "total_spent",
    "number_orders",
    "avg_order_value",
    "avg_delivery_delay"
]

# Prétraitement : log1p + standardisation
X_log = np.log1p(entities_pd[features])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

# Entraîner K-Means
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

# Examiner les médianes réelles de chaque cluster
print("Nombre de clients par cluster :")
print(entities_pd["cluster_id"].value_counts().sort_index())

print("\nMédianes par cluster :")
print(
    entities_pd.groupby("cluster_id")[features]
    .median()
    .round(2)
)

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
                                                                                

Nombre de clients par cluster :
cluster_id
0    31352
1    26552
2    32645
3     2801
Name: count, dtype: int64

Médianes par cluster :
            total_spent  number_orders  avg_order_value  avg_delivery_delay
cluster_id                                                                 
0                102.03            1.0           102.03                16.0
1                238.69            1.0           238.69                10.0
2                 58.16            1.0            58.16                 7.0
3                225.55            2.0           109.96                11.0


In [17]:
segment_names = {
    0: "Acheteurs intermédiaires",
    1: "Gros acheteurs occasionnels",
    2: "Petits acheteurs",
    3: "Clients récurrents"
}

entities_pd["segment"] = (
    entities_pd["cluster_id"].map(segment_names)
)

print("Répartition des segments :")
print(entities_pd["segment"].value_counts())

print("\nExemple de clients segmentés :")
print(
    entities_pd[
        ["customer_unique_id", "cluster_id", "segment"]
        + features
    ].head(10)
)

Répartition des segments :
segment
Petits acheteurs               32645
Acheteurs intermédiaires       31352
Gros acheteurs occasionnels    26552
Clients récurrents              2801
Name: count, dtype: int64

Exemple de clients segmentés :
                 customer_unique_id  cluster_id                   segment  \
0  000c8bdb58a29e7115cfc257230fb21b           2          Petits acheteurs   
1  a695332469eaed4d7e8af674d1c0a7fb           2          Petits acheteurs   
2  aefbb459b35ac7289f8ed198bb4e5a18           0  Acheteurs intermédiaires   
3  71430416a809573ed4a01e04fa8e3b91           2          Petits acheteurs   
4  762efb8c186a318d9d711692c0030c85           0  Acheteurs intermédiaires   
5  c2676cdedacba0e3aed5312e06b9bdda           2          Petits acheteurs   
6  45ba2acc9e8cbf07b686d5994d7592ab           2          Petits acheteurs   
7  4f617439bea51df96df53800f5b19aca           2          Petits acheteurs   
8  505953fd10284b83b2bd312099bdb160           0  Acheteurs intermé

In [18]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/olist_customer_segments"

(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)

print(
    f"Table de segmentation enregistrée dans "
    f"{GOLD_SEGMENTS_PATH} "
    f"({len(entities_pd):,} clients) !"
)

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/19 17:04:48 WARN TaskSetManager: Stage 109 cont

Table de segmentation enregistrée dans /lakehouse/gold/olist_customer_segments (93,350 clients) !


In [19]:
df_segments = spark.read.format("delta").load(
    "hdfs://namenode:9000/lakehouse/gold/olist_customer_segments"
)

print("Nombre de clients :", df_segments.count())

df_segments.groupBy("segment") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

Nombre de clients : 93350


[Stage 125:>                                                        (0 + 2) / 2]

+--------------------+-----+
|             segment|count|
+--------------------+-----+
|    Petits acheteurs|32645|
|Acheteurs intermé...|31352|
|Gros acheteurs oc...|26552|
|  Clients récurrents| 2801|
+--------------------+-----+



In [21]:
for table in [
    "gold_monthly_evolution",
    "gold_top_month_by_status",
    "olist_customer_segments"
]:
    print(f"\n===== {table} =====")
    df = spark.read.format("delta").load(
        f"hdfs://namenode:9000/lakehouse/gold/{table}"
    )
    df.printSchema()
    df.show(3, truncate=False)


===== gold_monthly_evolution =====
root
 |-- periode: string (nullable = true)
 |-- montant_total: double (nullable = true)
 |-- montant_precedent: double (nullable = true)
 |-- evolution: double (nullable = true)
 |-- taux_evolution_pct: double (nullable = true)



+-------+-------------+-----------------+-------------------+------------------+
|periode|montant_total|montant_precedent|evolution          |taux_evolution_pct|
+-------+-------------+-----------------+-------------------+------------------+
|2016-09|354.75       |NULL             |NULL               |NULL              |
|2016-10|56808.84     |354.75           |56454.09           |15913.77          |
|2016-12|19.62        |56808.84         |-56789.219999999994|-99.97            |
+-------+-------------+-----------------+-------------------+------------------+
only showing top 3 rows

===== gold_top_month_by_status =====
root
 |-- order_status: string (nullable = true)
 |-- periode: string (nullable = true)
 |-- montant_total: double (nullable = true)



+------------+-------+------------------+
|order_status|periode|montant_total     |
+------------+-------+------------------+
|delivered   |2017-11|1153364.2000000011|
|shipped     |2018-03|20620.66          |
|canceled    |2018-07|18944.45          |
+------------+-------+------------------+
only showing top 3 rows

===== olist_customer_segments =====
root
 |-- customer_unique_id: string (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- number_orders: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_delivery_delay: double (nullable = true)
 |-- cluster_id: long (nullable = true)
 |-- segment: string (nullable = true)



[Stage 163:>                                                        (0 + 1) / 1]

+--------------------------------+------------------+-------------+------------------+------------------+----------+---------------------------+
|customer_unique_id              |total_spent       |number_orders|avg_order_value   |avg_delivery_delay|cluster_id|segment                    |
+--------------------------------+------------------+-------------+------------------+------------------+----------+---------------------------+
|7a43947421cb96e7abe1e0572ccac62b|41.69             |1            |41.69             |12.0              |2         |Petits acheteurs           |
|013689381e404bf6304f57bab4b8f01d|392.87999999999994|1            |392.87999999999994|5.0               |1         |Gros acheteurs occasionnels|
|372bf744ad9bc73879c7e0ff2ce8760a|77.22999999999999 |1            |77.22999999999999 |9.0               |2         |Petits acheteurs           |
+--------------------------------+------------------+-------------+------------------+------------------+----------+--------------